[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/17_feature_engineering/04_end_to_end_pipeline.ipynb)

# Bài 04 — Pipeline feature engineering end-to-end

**🎯 Mục tiêu**
- Viết **transformer tuỳ chỉnh** (`BaseEstimator` + `TransformerMixin`) cho feature miền, có `get_feature_names_out`.
- Ghép mọi bước (làm sạch → feature miền → encoding → scaling → model) vào **một Pipeline** dùng được trong CV, GridSearch, và triển khai.
- Đo đóng góp của từng nhóm feature (ablation); lưu/tải pipeline bằng `joblib`.
- Checklist feature engineering.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/17_feature_engineering

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler

from ml_course.datasets import make_house_prices

houses = make_house_prices(n=4000, seed=12, dirty=True)
print(houses.shape)
houses.sample(5, random_state=0)

## 1. Transformer tuỳ chỉnh

Quy ước scikit-learn: `fit(X, y=None)` học tham số từ train (kết thúc bằng `_`), `transform(X)` áp dụng, không thay đổi `X` gốc; `__init__` chỉ lưu tham số.

In [ ]:
class HouseCleaner(TransformerMixin, BaseEstimator):
    """Làm sạch: chuẩn hoá tên quận, sửa diện tích bị gõ thừa số 0 (dựa trên trung vị học từ train)."""

    def __init__(self, area_factor=6.0):
        self.area_factor = area_factor

    def fit(self, X, y=None):
        self.area_median_ = np.nanmedian(X["area_m2"])
        return self

    def transform(self, X):
        X = X.copy()
        X["district"] = X["district"].str.strip().str.replace("Q.", "Quận", regex=False).str.title()
        too_big = X["area_m2"] > self.area_factor * self.area_median_
        X.loc[too_big, "area_m2"] = X.loc[too_big, "area_m2"] / 10          # lỗi gõ thừa số 0
        X["has_garden"] = X["has_garden"].astype(float)
        return X


class HouseFeatures(TransformerMixin, BaseEstimator):
    """Feature miền: log diện tích, diện tích / phòng ngủ, phòng tắm / phòng ngủ, nhà cũ."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["log_area"] = np.log(X["area_m2"])
        X["area_per_bedroom"] = X["area_m2"] / X["bedrooms"]
        X["bath_per_bed"] = X["bathrooms"] / X["bedrooms"]
        X["is_old"] = (X["age_years"] > 30).astype(float)
        return X


sample = HouseFeatures().fit_transform(HouseCleaner().fit_transform(houses.drop(columns="price_billion_vnd")))
sample[["district", "area_m2", "log_area", "area_per_bedroom", "bath_per_bed", "is_old"]].head()

## 2. Pipeline đầy đủ

In [ ]:
num_cols = ["area_m2", "log_area", "bedrooms", "bathrooms", "age_years", "distance_to_center_km", "area_per_bedroom", "bath_per_bed",
            "is_old", "has_garden"]
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("age_spline", make_pipeline(SimpleImputer(strategy="median"), SplineTransformer(n_knots=5)), ["age_years"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["district"]),
], verbose_feature_names_out=False)

linear_pipe = Pipeline([
    ("clean", HouseCleaner()),
    ("features", HouseFeatures()),
    ("preprocess", preprocess),
    ("model", TransformedTargetRegressor(RidgeCV(alphas=np.logspace(-3, 3, 13)), func=np.log, inverse_func=np.exp)),
])
X = houses.drop(columns="price_billion_vnd")
y = houses["price_billion_vnd"]
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
cv = KFold(5, shuffle=True, random_state=0)
mae = -cross_val_score(linear_pipe, X_dev, y_dev, cv=cv, scoring="neg_mean_absolute_error").mean()
print(f"Pipeline tuyến tính đầy đủ: MAE (CV) = {mae:.3f} tỉ")
linear_pipe.fit(X_dev, y_dev)
print("số feature sau tiền xử lý:", len(linear_pipe.named_steps["preprocess"].get_feature_names_out()))

Mọi bước — kể cả làm sạch dựa trên phân vị — đều được `fit` lại trên train của từng fold → không rò rỉ. Pipeline nhận **dữ liệu thô** (bẩn) và trả về dự đoán: chính là thứ cần triển khai.

## 3. Ablation: mỗi nhóm feature đóng góp bao nhiêu?

In [ ]:
def make_pipe(use_clean=True, use_features=True, log_target=True):
    cols = num_cols if use_features else ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km", "has_garden"]
    pre = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), cols),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), ["district"])])
    model = RidgeCV(alphas=np.logspace(-3, 3, 13))
    if log_target:
        model = TransformedTargetRegressor(model, func=np.log, inverse_func=np.exp)
    steps = []
    steps.append(("clean", HouseCleaner() if use_clean else FunctionLikeCast()))
    if use_features:
        steps.append(("features", HouseFeatures()))
    return Pipeline(steps + [("preprocess", pre), ("model", model)])


class FunctionLikeCast(TransformerMixin, BaseEstimator):
    """Chỉ ép kiểu has_garden (không làm sạch) — để so sánh."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.assign(has_garden=X["has_garden"].astype(float))


rows = []
for name, kw in [("feature thô, không làm sạch", dict(use_clean=False, use_features=False, log_target=False)),
                 ("+ làm sạch", dict(use_clean=True, use_features=False, log_target=False)),
                 ("+ log target", dict(use_clean=True, use_features=False, log_target=True)),
                 ("+ feature miền (đầy đủ)", dict(use_clean=True, use_features=True, log_target=True))]:
    rows.append({"cấu hình": name, "MAE (CV)": -cross_val_score(make_pipe(**kw), X_dev, y_dev, cv=cv, scoring="neg_mean_absolute_error").mean()})
pd.DataFrame(rows).round(3)

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
ablation = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8.5, 3))
ax.barh(ablation["cấu hình"][::-1], ablation["MAE (CV)"][::-1], color="C0")
for i, v in enumerate(ablation["MAE (CV)"][::-1]):
    ax.text(v, i, f" {v:.3f}", va="center")
ax.set_xlabel("MAE (CV, tỉ đồng) — thấp hơn là tốt hơn")
ax.set_title("Ablation: thêm dần từng bước")
plt.tight_layout()
plt.show()

Làm sạch dữ liệu đóng góp nhiều nhất. Để ý: **log target một mình không giúp** (thậm chí tệ hơn chút) khi diện tích vẫn ở thang tuyến tính — quan hệ nhân chỉ thành tuyến tính khi log **cả hai vế** (log giá ~ log diện tích): các biến đổi phải đi cùng nhau theo hiểu biết về cấu trúc bài toán.

## 4. So với gradient boosting

In [ ]:
gb_pre = ColumnTransformer([("num", "passthrough", num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), ["district"])])
gb_pipe = Pipeline([("clean", HouseCleaner()), ("features", HouseFeatures()), ("preprocess", gb_pre),
                    ("model", HistGradientBoostingRegressor(random_state=0))])
gb_mae = -cross_val_score(gb_pipe, X_dev, y_dev, cv=cv, scoring="neg_mean_absolute_error").mean()
print(f"HistGB + cùng feature: MAE (CV) = {gb_mae:.3f} tỉ | Ridge pipeline: {mae:.3f} tỉ")

## 5. Đánh giá cuối và lưu pipeline

In [ ]:
from pathlib import Path
from sklearn.metrics import mean_absolute_error

best = linear_pipe if mae <= gb_mae else gb_pipe.fit(X_dev, y_dev)
test_mae = mean_absolute_error(y_test, best.predict(X_test))
print(f"pipeline được chọn: {best.named_steps['model'].__class__.__name__} | MAE test = {test_mae:.3f} tỉ")

out = Path("../../assets/house_price_pipeline.joblib")
out.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best, out)
reloaded = joblib.load(out)
new_house = pd.DataFrame([{"area_m2": 80.0, "bedrooms": 3, "bathrooms": 2, "age_years": 5, "district": " quận 7 ",
                           "distance_to_center_km": 7.5, "has_garden": False}])
print(f"dự đoán cho căn nhà mới (dữ liệu thô, tên quận 'bẩn'): {reloaded.predict(new_house)[0]:.2f} tỉ")
out.unlink()                                  # dọn file tạm

> ⚠️ Pickle/joblib chứa **code** của class tuỳ chỉnh theo tên module — khi tải lại cần cùng phiên bản thư viện và định nghĩa class. Trong dự án thật, đặt transformer tuỳ chỉnh vào một package (vd `src/`) thay vì notebook (chương 39).

## 6. Checklist feature engineering

1. Hiểu dữ liệu và bài toán (EDA, chuyên gia miền) trước khi tạo feature.
2. Chỉ dùng thông tin **có sẵn tại thời điểm dự đoán** (không rò rỉ tương lai/target).
3. Bắt đầu với baseline đơn giản; thêm feature theo giả thuyết; đo bằng CV (ablation).
4. Mọi bước học từ dữ liệu nằm trong Pipeline (fit trên train của từng fold).
5. Biến đổi phù hợp với model (scaling/log cho tuyến tính; tỉ số/aggregation cho mọi model).
6. Xử lý category theo cardinality; target encoding luôn cross-fit.
7. Ghi lại định nghĩa từng feature (feature dictionary) để tái lập và triển khai (feature store — chương 39).

## 🧠 Kiểm tra nhanh

1. Vì sao transformer làm sạch dựa trên phân vị phải nằm trong Pipeline thay vì chạy trước trên toàn bộ dữ liệu?
2. Quy ước của `fit` và `transform` trong một transformer scikit-learn là gì?
3. Ablation study dùng để làm gì?
4. Rủi ro khi lưu pipeline chứa class tuỳ chỉnh định nghĩa trong notebook?

## 📌 Tóm tắt

- Transformer tuỳ chỉnh + ColumnTransformer + Pipeline = một đối tượng nhận dữ liệu thô, dùng được cho CV, tuning, triển khai.
- Ablation đo đóng góp từng bước; feature engineering tốt giúp model đơn giản cạnh tranh với model phức tạp.
- Lưu pipeline (joblib) — chú ý phụ thuộc code.

## 📚 Tài liệu tham khảo

- scikit-learn — Developing scikit-learn estimators
- Huyen (2022) — *Designing Machine Learning Systems*, chương 5 (Feature Engineering)